# 🧬 EENP 1.0 — All 12 Layers in Google Colab
**Corrected, expanded notebook.** Every layer below runs directly in Colab. Open the left-hand table of contents to navigate Layers 01–12. No Streamlit or ngrok token is needed for these layers.

Run Setup, then Layers 01–12 in order. When changing data or context, rerun the subsequent layers. CPU is sufficient. All default evidence is synthetic. Outputs are exploratory, not validated biological predictions. Several advanced functions in the poster are represented by explicitly labeled simplified analyses; there is no trained AI model.

## Setup · Install dependencies

In [ ]:
%pip -q install "streamlit>=1.40,<2" "pandas>=2.2,<3" "numpy>=1.26,<3" "networkx>=3.2,<4" "plotly>=5.24,<7" "pyngrok>=7.2,<9"

## Setup · Create source files
This embedded source also provides the optional Streamlit app.

In [ ]:
from pathlib import Path
import os, sys
project = Path("/content/EENP")
project.mkdir(exist_ok=True)
files = {'core.py': '"""EENP exploratory network engine. All default evidence is synthetic."""\nimport io, json, hashlib, zipfile, platform\nimport numpy as np\nimport pandas as pd\nimport networkx as nx\nimport plotly.graph_objects as go\n\nCONTEXT = [\'tissue\',\'species\',\'disease\']\nREQUIRED = [\'source\',\'target\',\'source_domain\',\'target_domain\',\'effect\',\'confidence\',\'reference\',\'method\'] + CONTEXT\n\ndef demo():\n    pairs = [(\'Compound_A\',\'Gene_A\',\'exogenous\',\'endogenous\',1,.9),(\'Compound_A\',\'Gene_B\',\'exogenous\',\'endogenous\',-1,.8),(\'Exposure_B\',\'Gene_B\',\'exogenous\',\'endogenous\',1,.7),(\'Gene_A\',\'Protein_A\',\'endogenous\',\'endogenous\',1,.95),(\'Gene_B\',\'Protein_A\',\'endogenous\',\'endogenous\',-1,.75),(\'Protein_A\',\'Pathway_A\',\'endogenous\',\'endogenous\',1,.85),(\'Pathway_A\',\'Outcome_A\',\'endogenous\',\'endogenous\',1,.8),(\'Protein_A\',\'Compound_A\',\'endogenous\',\'exogenous\',-1,.6),(\'Compound_A\',\'Gene_A\',\'exogenous\',\'endogenous\',-1,.4),(\'Exposure_B\',\'Pathway_A\',\'exogenous\',\'endogenous\',-1,.55)]\n    rows=[]\n    for tissue in [\'Tissue_A\',\'Tissue_B\']:\n        for i,p in enumerate(pairs):\n            s,t,sd,td,e,c=p\n            if tissue==\'Tissue_B\' and i in [2,7]: continue\n            rows.append([s,t,sd,td, -e if tissue==\'Tissue_B\' and i==3 else e,c,\'SYNTHETIC-\'+str(i),\'synthetic\',tissue,\'demo_species\',\'demo_condition\'])\n    return pd.DataFrame(rows,columns=REQUIRED)\n\ndef validate(df):\n    missing=set(REQUIRED)-set(df.columns)\n    if missing: raise ValueError(\'Missing columns: \'+\', \'.join(sorted(missing)))\n    if df.empty: raise ValueError(\'The evidence file is empty.\')\n    if len(df)>5000: raise ValueError(\'Prototype limit: 5,000 evidence rows.\')\n    df=df.copy()\n    for c in REQUIRED:\n        if df[c].isna().any(): raise ValueError(f\'{c}: missing values are not allowed.\')\n        if c not in [\'effect\',\'confidence\']:\n            df[c]=df[c].astype(str).str.strip()\n            if df[c].eq(\'\').any(): raise ValueError(f\'{c}: blank values are not allowed.\')\n    for c in [\'effect\',\'confidence\']: df[c]=pd.to_numeric(df[c],errors=\'raise\')\n    if not df.effect.isin([-1,1]).all(): raise ValueError(\'effect must be +1 or -1; unknown direction must be curated before import.\')\n    if not np.isfinite(df.confidence).all() or not df.confidence.between(0,1).all(): raise ValueError(\'confidence must be finite and in [0,1].\')\n    domains={}\n    for side in [\'source\',\'target\']:\n        for n,d in zip(df[side],df[side+\'_domain\']):\n            if d not in [\'endogenous\',\'exogenous\']: raise ValueError(\'Domains must be endogenous or exogenous.\')\n            if n in domains and domains[n]!=d: raise ValueError(f\'Conflicting domains for {n}.\')\n            domains[n]=d\n    if len(domains)>150: raise ValueError(\'Prototype limit: 150 nodes.\')\n    return df.drop_duplicates().reset_index(drop=True)\n\ndef select(df,context,threshold=0):\n    mask=df.confidence.ge(threshold)\n    for c,v in context.items(): mask &= df[c].eq(v)\n    return df.loc[mask].copy()\n\ndef aggregate(df):\n    rows=[]\n    for (s,t),g in df.groupby([\'source\',\'target\'],sort=True):\n        total=g.confidence.sum(); signed=(g.effect*g.confidence).sum()\n        rows.append(dict(source=s,target=t,weight=float(signed/len(g)),confidence=float(g.confidence.mean()),disagreement=float(1-abs(signed)/total) if total else 0.,evidence_count=len(g)))\n    return pd.DataFrame(rows,columns=[\'source\',\'target\',\'weight\',\'confidence\',\'disagreement\',\'evidence_count\'])\n\ndef graph(df):\n    g=nx.DiGraph()\n    for r in df.itertuples():\n        g.add_node(r.source,domain=r.source_domain);g.add_node(r.target,domain=r.target_domain)\n    for r in aggregate(df).itertuples(): g.add_edge(r.source,r.target,weight=r.weight)\n    return g\n\ndef network_figure(df):\n    g=graph(df);pos=nx.spring_layout(g,seed=42)\n    fig=go.Figure()\n    for a,b,d in g.edges(data=True):\n        x,y=pos[a];u,v=pos[b]\n        color=\'#0d9488\' if d[\'weight\']>0 else \'#e11d48\' if d[\'weight\']<0 else \'#94a3b8\'\n        fig.add_trace(go.Scatter(x=[x,u],y=[y,v],mode=\'lines\',line=dict(color=color,width=1+3*abs(d[\'weight\'])),hoverinfo=\'text\',text=f\'{a} → {b}: {d["weight"]:.2f}\',showlegend=False))\n        fig.add_annotation(x=u*.85+x*.15,y=v*.85+y*.15,ax=u*.65+x*.35,ay=v*.65+y*.35,xref=\'x\',yref=\'y\',axref=\'x\',ayref=\'y\',showarrow=True,arrowhead=2,arrowcolor=color,text=\'\')\n    for domain,color in [(\'endogenous\',\'#0d9488\'),(\'exogenous\',\'#9333ea\')]:\n        ns=[n for n in g if g.nodes[n][\'domain\']==domain]\n        fig.add_trace(go.Scatter(x=[pos[n][0] for n in ns],y=[pos[n][1] for n in ns],text=ns,mode=\'markers+text\',textposition=\'top center\',marker=dict(size=22,color=color,line=dict(width=2,color=\'white\')),name=domain))\n    fig.update_layout(height=550,template=\'plotly_white\',xaxis=dict(visible=False),yaxis=dict(visible=False),margin=dict(l=20,r=20,t=25,b=20),legend=dict(orientation=\'h\'))\n    return fig\n\ndef simulate(df,interventions,steps=30,pulse=10,gain=.6):\n    g=graph(df);nodes=sorted(g);idx={n:i for i,n in enumerate(nodes)}\n    w=np.zeros((len(nodes),len(nodes)))\n    for a,b,d in g.edges(data=True): w[idx[b],idx[a]]=d[\'weight\']\n    w=w/max(1.,np.abs(w).sum(axis=1).max(initial=0))\n    x=np.zeros(len(nodes));u=np.zeros(len(nodes))\n    for n,v in interventions.items():\n        if n not in idx: raise ValueError(\'Intervention node missing: \'+n)\n        u[idx[n]]=v\n    result=[x.copy()]\n    for t in range(steps):\n        x=np.tanh(gain*w@x+(u if t<pulse else 0));result.append(x.copy())\n    return pd.DataFrame(result,columns=nodes).rename_axis(\'step\')\n\ndef paths(df,source,target,max_hops=5):\n    g=graph(df);rows=[]\n    # Bounded depth and output for predictable interactive runtime.\n    for p in nx.all_simple_paths(g,source,target,cutoff=max_hops):\n        ws=[g[a][b][\'weight\'] for a,b in zip(p,p[1:])]\n        rows.append({\'path\':\' → \'.join(p),\'signed_product\':float(np.prod(ws)),\'hops\':len(ws)})\n        if len(rows)>=100: break\n    return pd.DataFrame(rows)\n\ndef package(df,context,params,trajectory):\n    buf=io.BytesIO();raw=df.to_csv(index=False)\n    manifest={\'prototype\':\'EENP 1.0 research prototype\',\'context\':context,\'parameters\':params,\'evidence_sha256\':hashlib.sha256(raw.encode()).hexdigest(),\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\'networkx\':nx.__version__,\'limitations\':\'Synthetic default data. No trained AI or clinical validation. Context transfer is edge overlap, not predictive validation. Simulation units are arbitrary.\'}\n    with zipfile.ZipFile(buf,\'w\',zipfile.ZIP_DEFLATED) as z:\n        z.writestr(\'evidence.csv\',raw);z.writestr(\'aggregated_edges.csv\',aggregate(df).to_csv(index=False));z.writestr(\'simulation.csv\',trajectory.to_csv());z.writestr(\'manifest.json\',json.dumps(manifest,indent=2))\n    return buf.getvalue()\n', 'app.py': 'import json\nimport pandas as pd\nimport plotly.express as px\nimport streamlit as st\nfrom core import *\n\nst.set_page_config(page_title=\'EENP 1.0 | Research Studio\',page_icon=\'🧬\',layout=\'wide\')\nst.markdown(\'\'\'<style>\n.block-container {padding-top:2rem} \n.hero {background:linear-gradient(115deg,#123e84,#078b91 55%,#7c3aed);padding:30px;border-radius:24px;color:white;margin-bottom:22px;box-shadow:0 12px 30px #123e8422}\n.hero h1 {color:white;font-size:2.8rem;margin:0}.hero p{color:#e0f2fe;font-size:1.1rem}\n[data-testid="stMetric"]{background:white;padding:18px;border-radius:16px;border-top:4px solid #14b8a6;box-shadow:0 4px 18px #1725540a}\n</style><div class="hero"><h1>🧬 EENP 1.0</h1><p>Endogenous–Exogenous Interaction Network • Research Studio</p>INTEGRATE &nbsp; • &nbsp; CONNECT &nbsp; • &nbsp; DISCOVER &nbsp; • &nbsp; EXPLORE</div>\'\'\',unsafe_allow_html=True)\nst.caption(\'Concept based on the supplied EENP poster. Independent computational prototype; not an implementation validated by its inventor.\')\nst.warning(\'Research prototype • Default data are entirely synthetic • No trained AI, clinical predictions, or treatment recommendations.\')\nmodules=[\'01 · Biological context\',\'02 · Identity & data harmonization\',\'03 · Evidence & provenance\',\'04 · Dual-state network\',\'05 · Cross-domain bridges\',\'06 · Interaction asymmetry\',\'07 · Competing mechanisms\',\'08 · Perturbation & recovery\',\'09 · Uncertainty\',\'10 · Next evidence\',\'11 · Cross-context comparison\',\'12 · Reproducible package\']\nwith st.sidebar:\n    st.title(\'🔬 Research workspace\')\n    page=st.radio(\'Architecture\',modules)\n    mode=st.radio(\'Evidence source\',[\'Synthetic demo\',\'Upload CSV\'])\n    upload=st.file_uploader(\'Evidence CSV\',type=[\'csv\']) if mode==\'Upload CSV\' else None\n    st.download_button(\'Download example CSV\',demo().to_csv(index=False),\'example_evidence.csv\',\'text/csv\')\ntry:\n    if mode==\'Upload CSV\' and upload is None:\n        st.info(\'Upload an evidence CSV or choose Synthetic demo.\');st.stop()\n    raw=validate(pd.read_csv(upload) if upload is not None else demo())\nexcept Exception as exc:\n    st.error(str(exc));st.stop()\nwith st.sidebar:\n    context={}\n    remaining=raw\n    for c in CONTEXT:\n        context[c]=st.selectbox(c.title(),sorted(remaining[c].unique()))\n        remaining=remaining[remaining[c].eq(context[c])]\n    threshold=st.slider(\'Minimum evidence confidence\',0.,1.,.3,.05)\n    st.caption(\'Confidence values are supplied by the data author; they are not calibrated probabilities.\')\ndf=select(raw,context,threshold)\nif df.empty:\n    st.info(\'No evidence passes these filters. Lower the confidence threshold.\');st.stop()\ng=graph(df);edges=aggregate(df);nodes=sorted(g)\ncols=st.columns(4)\nfor col,label,val in zip(cols,[\'Entities\',\'Evidence records\',\'Directed connections\',\'Cross-domain links\'],[len(g),len(df),len(edges),sum(g.nodes[a][\'domain\']!=g.nodes[b][\'domain\'] for a,b in g.edges)]): col.metric(label,val)\nst.subheader(page)\nif page==modules[0]:\n    st.json(context)\n    st.info(\'Filters match tissue, species and disease exactly. Dose, time, cell type and genotype are recorded below as study metadata only; this prototype does not infer their effects.\')\n    for field in [\'Cell type\',\'Genotype\',\'Exposure / dose and units\',\'Observation time and units\']:\n        st.text_input(field,key=\'meta_\'+field)\nelif page==modules[1]:\n    st.write(\'Whitespace is trimmed, exact duplicates removed, numeric ranges checked, and conflicting entity domains rejected. IDs remain case-sensitive.\')\n    st.info(\'Use curated canonical identifiers. Automatic gene/protein synonym resolution, ontology mapping and species orthology are not implemented.\')\n    st.dataframe(raw,use_container_width=True)\n    st.code(\', \'.join(REQUIRED))\nelif page==modules[2]:\n    st.dataframe(df,use_container_width=True)\n    st.plotly_chart(px.histogram(df,x=\'method\',color=\'source_domain\',color_discrete_sequence=[\'#0d9488\',\'#9333ea\']),use_container_width=True)\n    st.caption(\'References are preserved as supplied; their authenticity is not automatically verified.\')\nelif page==modules[3]:\n    st.plotly_chart(network_figure(df),use_container_width=True)\n    st.caption(\'Arrows indicate direction. Teal edges: positive net effect; pink: negative; gray: cancellation. Zero-weight edges remain evidence connections.\')\nelif page==modules[4]:\n    cross=edges[[g.nodes[r.source][\'domain\']!=g.nodes[r.target][\'domain\'] for r in edges.itertuples()]]\n    st.dataframe(cross,use_container_width=True)\n    st.caption(\'Direct bridges between domains in the selected evidence. Explore indirect connections in Competing mechanisms.\')\nelif page==modules[5]:\n    rows=[]\n    for r in edges.itertuples():\n        reverse=g.get_edge_data(r.target,r.source)\n        rows.append({\'source\':r.source,\'target\':r.target,\'forward_weight\':r.weight,\'reverse_weight\':reverse[\'weight\'] if reverse else None,\'reverse_evidence\':\'present\' if reverse else \'missing\'})\n    st.dataframe(pd.DataFrame(rows),use_container_width=True)\n    st.info(\'Missing reverse evidence does not prove the reverse biological effect is absent.\')\nelif page==modules[6]:\n    a,b=st.columns(2)\n    source=a.selectbox(\'Starting entity\',nodes);target=b.selectbox(\'Destination entity\',nodes,index=len(nodes)-1)\n    hops=st.slider(\'Maximum pathway length\',1,6,4)\n    if source==target: st.info(\'Choose two different entities. Feedback cycles are not enumerated.\')\n    else:\n        result=paths(df,source,target,hops)\n        if result.empty: st.info(\'No directed path within this length limit.\')\n        else: st.dataframe(result,use_container_width=True)\n    st.caption(\'Up to 100 simple paths in traversal order. Signed products are heuristic model weights, not mechanism probabilities or causal proof.\')\nelif page==modules[7]:\n    selected=st.multiselect(\'Perturb entities\',nodes,default=[nodes[0]])\n    interventions={n:st.slider(\'Input: \'+n,-1.,1.,.5,.1) for n in selected}\n    pulse=st.slider(\'Input duration (steps)\',1,30,10)\n    gain=st.slider(\'Propagation gain\',0.,.95,.6,.05)\n    trajectory=simulate(df,interventions,40,pulse,gain)\n    st.plotly_chart(px.line(trajectory,labels={\'value\':\'Model response (arbitrary units)\',\'index\':\'Step\'},color_discrete_sequence=px.colors.qualitative.Vivid),use_container_width=True)\n    st.caption(\'Zero initial state and zero unperturbed baseline. Input stops after the chosen duration. Multiple inputs are simulated together. This is a bounded toy dynamical system, not a biological dose-response model.\')\n    st.session_state[\'last_sim\']={\'context\':context,\'threshold\':threshold,\'interventions\':interventions,\'steps\':40,\'pulse\':pulse,\'gain\':gain}\n    st.latex(r\'x_{t+1}=\\tanh(g\\,\\widetilde{W}x_t+u_t)\')\nelif page==modules[8]:\n    st.dataframe(edges.sort_values(\'disagreement\',ascending=False),use_container_width=True)\n    st.plotly_chart(px.scatter(edges,x=\'confidence\',y=\'disagreement\',size=\'evidence_count\',hover_name=\'source\',color=\'disagreement\',color_continuous_scale=\'Plasma\'),use_container_width=True)\n    st.caption(\'Disagreement = 1 − |sum(signed confidence)| / sum(confidence). This measures directional conflict, not statistical uncertainty. Zero-confidence records supply no directional information.\')\nelif page==modules[9]:\n    ranked=edges.copy()\n    ranked[\'priority\']=(1-ranked.confidence+ranked.disagreement)/(1+ranked.evidence_count)\n    st.dataframe(ranked.sort_values(\'priority\',ascending=False),use_container_width=True)\n    st.info(\'Heuristic prioritization of existing links only: (1 − mean confidence + disagreement) / (1 + record count). Cost, feasibility, missing links and expected information gain are not modeled.\')\nelif page==modules[10]:\n    alternative={}\n    for c in CONTEXT: alternative[c]=st.selectbox(\'Comparison \'+c,sorted(raw[c].unique()),key=\'alt_\'+c)\n    other=aggregate(select(raw,alternative,threshold))\n    if alternative==context: st.info(\'Select a different context to make a meaningful comparison.\')\n    if other.empty: st.info(\'No matching comparison evidence.\')\n    else:\n        merged=edges.merge(other,on=[\'source\',\'target\'],how=\'outer\',suffixes=(\'_current\',\'_comparison\'),indicator=True)\n        intersection=int(merged[\'_merge\'].eq(\'both\').sum())\n        st.metric(\'Directed edge Jaccard overlap\',f\'{intersection/len(merged):.2f}\')\n        st.dataframe(merged,use_container_width=True)\n        st.caption(\'Descriptive overlap and weight comparison only. No cross-context predictive transfer validation is performed.\')\nelse:\n    params=st.session_state.get(\'last_sim\',{\'context\':context,\'threshold\':threshold,\'interventions\':{nodes[0]:.5},\'steps\':40,\'pulse\':10,\'gain\':.6})\n    if params[\'context\']!=context or params[\'threshold\']!=threshold or any(n not in g for n in params[\'interventions\']):\n        params={\'context\':context,\'threshold\':threshold,\'interventions\':{nodes[0]:.5},\'steps\':40,\'pulse\':10,\'gain\':.6}\n        st.info(\'Filters changed. Export uses the displayed default simulation settings.\')\n    st.json(params)\n    result=simulate(df,params[\'interventions\'],params[\'steps\'],params[\'pulse\'],params[\'gain\'])\n    metadata={k:v for k,v in st.session_state.items() if k.startswith(\'meta_\')}\n    st.download_button(\'📦 Download reproducible evidence package\',package(df,context,{**params,\'study_metadata\':metadata},result),\'EENP_results.zip\',\'application/zip\')\n    st.download_button(\'Download complete imported evidence\',raw.to_csv(index=False),\'all_evidence.csv\',\'text/csv\')\n    st.caption(\'ZIP includes filtered evidence, aggregated edges, simulation, parameters, software versions and evidence checksum. Keep the source-code bundle alongside it.\')\n', 'requirements.txt': 'streamlit>=1.40,<2\npandas>=2.2,<3\nnumpy>=1.26,<3\nnetworkx>=3.2,<4\nplotly>=5.24,<7\n', 'README.md': '# EENP 1.0 — colorful research prototype\n\nBased on the 12-layer concept in the supplied poster. This is an independent exploratory implementation, not a scientifically validated reproduction. All default entities, contexts and evidence are synthetic. No API key required for analysis. No trained AI model, clinical prediction, literature retrieval or treatment recommendations are included.\n\n## Start in Google Colab\n1. Open https://colab.research.google.com/ and choose File → Upload notebook.\n2. Upload EENP_Colab.ipynb, included in this ZIP.\n3. Run cells from the top. The notebook writes the complete app and engine itself; no ZIP upload is needed.\n4. The analysis works directly in the notebook. Optional final cells launch Streamlit using ngrok. For these, create an ngrok account and add NGROK_AUTHTOKEN to Colab Secrets (key icon). Enable notebook access. The resulting URL is temporary and public; use synthetic/non-sensitive data. Stop the tunnel when finished.\n\n## Start Streamlit locally (Python 3.10+)\nUnzip this package, open a terminal inside the EENP folder, then run:\n\n    python -m pip install -r requirements.txt\n    python -m streamlit run app.py\n\nOpen the localhost URL printed by Streamlit. For Streamlit Community Cloud, upload app.py, core.py, requirements.txt and .streamlit/config.toml to a GitHub repository; choose app.py as the entry point. This project does not contain user authentication or a patient-data backend.\n\n## Your own evidence\nSelect Upload CSV in the sidebar. Start from example_evidence.csv. Each row is one directed evidence record. Required columns:\n\n- source, target: canonical entity identifiers (case-sensitive).\n- source_domain, target_domain: endogenous or exogenous.\n- effect: 1 for positive, -1 for negative; curate unknown effects separately.\n- confidence: author-supplied score from 0 to 1, not a calibrated probability.\n- reference: traceable publication/database identifier; not automatically verified.\n- method: experimental/computational method or synthetic.\n- tissue, species, disease: exact context labels.\n\nOptional columns are preserved. Limits: 5,000 rows and 150 nodes. Exact duplicate rows are removed; other records are not assumed independent. Curate correlated studies before interpretation. Context filters match exactly. No inferred synonym/ortholog mapping. Dose, time, genotype and cell type are metadata only.\n\n## Model and layer scope\n1 Context: exact tissue/species/disease filters plus metadata.\n2 Harmonization: whitespace, duplicates, schema, domains and range checks; no automatic ID mapping.\n3 Provenance: row-level evidence and reference display; no source verification.\n4 Dual-state graph: directed network with endogenous/exogenous node colors.\n5 Bridges: direct cross-domain links; indirect paths in layer 7.\n6 Asymmetry: forward and reverse evidence, preserving missingness.\n7 Mechanisms: up to 100 simple paths, max 6 hops; no feedback-cycle enumeration.\n8 Simulation: zero-state tanh propagation with temporary, simultaneous inputs.\n9 Uncertainty: weighted directional disagreement; no calibrated intervals.\n10 Next evidence: explicit heuristic ranking; not experimental optimization.\n11 Transfer: descriptive context overlap; no predictive validation.\n12 Package: CSVs, manifest, checksum and software versions.\n\nFor each directed pair, weight = mean(effect × confidence). Opposing evidence may cancel. W[target, source] is divided by max(1, maximum absolute row sum). Dynamics: x[t+1] = tanh(gain × W × x[t] + input[t]); input is zero after pulse duration. Baseline is zero and model units are arbitrary. The gain is below one. This model does not predict real pharmacology or biological recovery.\n\n## Reproducibility\nSeed 42 fixes network layout. Results ZIP preserves parameters, selected evidence and software versions. requirements.txt contains compatible version ranges, not an exact lock; use pip freeze to record your deployed environment. Source code should be retained with each experiment.\n\nDocumentation:\nhttps://docs.streamlit.io/develop/api-reference/charts/st.plotly_chart\nhttps://docs.streamlit.io/develop/api-reference/widgets/st.download_button\nhttps://networkx.org/documentation/stable/reference/algorithms/simple_paths.html\n', '.streamlit/config.toml': '[theme]\nprimaryColor = "#7c3aed"\nbackgroundColor = "#f7f9ff"\nsecondaryBackgroundColor = "#edf2ff"\ntextColor = "#172554"\nfont = "sans serif"\n[server]\nmaxUploadSize = 10\n'}
for name, content in files.items():
    target = project / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
os.chdir(project)
if str(project) not in sys.path: sys.path.insert(0, str(project))
print("EENP source files are ready.")

## Setup · Load evidence
Set USE_UPLOAD to True to upload exactly one CSV. The default dataset demonstrates every analysis without external credentials.

In [ ]:
from core import *
from IPython.display import display, HTML
import plotly.express as px
import plotly.io as pio
import html
pio.renderers.default = 'colab'
px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = px.colors.qualitative.Vivid

def banner(number, title):
    display(HTML(f'<div style="background:linear-gradient(110deg,#123e84,#0d9488,#7c3aed);color:white;padding:20px;border-radius:16px;margin:12px 0"><b style="font-size:23px">{number:02d} · {html.escape(title)}</b></div>'))

USE_UPLOAD = False
if USE_UPLOAD:
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    if len(uploaded) != 1: raise ValueError('Upload exactly one CSV.')
    loaded = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))
else:
    loaded = demo()
evidence = validate(loaded)
evidence.to_csv('example_evidence.csv', index=False)
print(f'Loaded {len(evidence)} valid evidence records.')
display(evidence.head())

## Layer 01 · Biological Context Fingerprint
Edit the context values below using the displayed available contexts. Dose, cell type, genotype and time are metadata, not model inputs.

In [ ]:
banner(1, 'Biological Context Fingerprint')
display(evidence[CONTEXT].drop_duplicates().reset_index(drop=True))
context = evidence[CONTEXT].iloc[0].to_dict()
# Example: context['tissue'] = 'Tissue_B'
threshold = 0.3
study_metadata = {'cell_type': 'unspecified', 'genotype': 'unspecified',
                  'dose_with_units': 'unspecified', 'observation_time_with_units': 'unspecified'}
selected = select(evidence, context, threshold)
if selected.empty: raise ValueError('No matching evidence. Edit context or lower threshold.')
context_fingerprint = hashlib.sha256(json.dumps({'context': context, 'metadata': study_metadata}, sort_keys=True).encode()).hexdigest()
display(pd.DataFrame([dict(**context, **study_metadata)]))
print('Context fingerprint:', context_fingerprint)
print('Evidence records passing filters:', len(selected))

## Layer 02 · Identity & Data Harmonization
Checks schema, whitespace, duplicates, domains and numeric ranges. Canonical identifiers must be curated by you; automatic synonyms, ontologies and ortholog mapping are not implemented.

In [ ]:
banner(2, 'Identity & Data Harmonization')
validation_summary = pd.DataFrame({'measure': ['Input records', 'Validated records', 'Exact duplicates removed'],
                                   'count': [len(loaded), len(evidence), len(loaded)-len(evidence)]})
display(validation_summary)
entity_table = pd.concat([
    evidence[['source','source_domain']].rename(columns={'source':'entity','source_domain':'domain'}),
    evidence[['target','target_domain']].rename(columns={'target':'entity','target_domain':'domain'})
]).drop_duplicates().sort_values('entity').reset_index(drop=True)
display(entity_table)
print('Required columns:', ', '.join(REQUIRED))

## Layer 03 · Evidence–Provenance Graph
Inspect supporting records and supplied references. References are preserved but not independently verified. Confidence values are author-supplied scores.

In [ ]:
banner(3, 'Evidence–Provenance Graph')
provenance = selected.copy()
provenance.insert(0, 'record_id', ['record_'+str(i+1) for i in range(len(provenance))])
display(provenance)
px.histogram(provenance, x='method', color='source_domain', title='Evidence by method and source domain').show()
print('Each directed pair may have multiple supporting or conflicting records.')

## Layer 04 · Dual-State Network Constructor
Directed graph combining endogenous and exogenous entities. Teal nodes are endogenous; purple are exogenous. Edges: teal positive, pink negative, gray cancellation.

In [ ]:
banner(4, 'Dual-State Network Constructor')
g = graph(selected)
edges = aggregate(selected)
entities = sorted(g)
network_figure(selected).show()
display(pd.DataFrame([{'nodes':len(g), 'directed_edges':len(edges), 'evidence_records':len(selected)}]))
display(edges)

## Layer 05 · Cross-Domain Bridge Discovery
Find direct links between domains and show indirect routes later in Layer 7. These are evidence connections, not newly validated biological discoveries.

In [ ]:
banner(5, 'Cross-Domain Bridge Discovery')
bridge_mask = [g.nodes[r.source]['domain'] != g.nodes[r.target]['domain'] for r in edges.itertuples()]
bridges = edges.loc[bridge_mask].copy()
bridges['direction'] = [g.nodes[r.source]['domain']+' → '+g.nodes[r.target]['domain'] for r in bridges.itertuples()]
display(bridges)
if not bridges.empty:
    px.bar(bridges.assign(link=bridges.source+' → '+bridges.target), x='link', y='weight', color='direction', title='Cross-domain bridge weights').show()
else: print('No direct cross-domain links in this context.')

## Layer 06 · Interaction Asymmetry Engine
Compare forward and reverse evidence. Missing reverse evidence does not establish biological absence.

In [ ]:
banner(6, 'Interaction Asymmetry Engine')
asymmetry_rows = []
for r in edges.itertuples():
    reverse = g.get_edge_data(r.target, r.source)
    reverse_weight = reverse['weight'] if reverse is not None else None
    asymmetry_rows.append({'source':r.source, 'target':r.target, 'forward_weight':r.weight,
                           'reverse_weight':reverse_weight, 'reverse_evidence':'present' if reverse is not None else 'missing',
                           'difference_when_observed':r.weight-reverse_weight if reverse is not None else None})
asymmetry = pd.DataFrame(asymmetry_rows)
display(asymmetry)

## Layer 07 · Competing Mechanism Generator
Enumerates up to 100 directed simple paths in traversal order, within the hop limit. Signed products are heuristic weights, not probabilities. Feedback loops are not enumerated.

In [ ]:
banner(7, 'Competing Mechanism Generator')
print('Available entities:', entities)
source, target = ('Compound_A', 'Outcome_A') if {'Compound_A','Outcome_A'} <= set(entities) else (entities[0], entities[-1])
max_hops = 5
if not 1 <= max_hops <= 6: raise ValueError('Choose max_hops from 1 to 6.')
if source not in g or target not in g: raise ValueError('Choose entities from the list above.')
mechanisms = paths(selected, source, target, max_hops) if source != target else pd.DataFrame()
display(mechanisms)
if mechanisms.empty: print('No path found; choose different entities or increase the hop limit.')

## Layer 08 · Perturbation & Recovery Simulator
A bounded toy model with simultaneous inputs, zero initial state and arbitrary response units. Dose-response and real biological recovery are not predicted. Change interventions to explore combinations.

In [ ]:
banner(8, 'Perturbation & Recovery Simulator')
intervention_node = 'Compound_A' if 'Compound_A' in g else entities[0]
interventions = {intervention_node: 0.5}
steps, pulse, gain = 40, 10, 0.6
if not 1 <= pulse <= steps <= 200: raise ValueError('Require 1 <= pulse <= steps <= 200.')
if not 0 <= gain < 1: raise ValueError('Gain must be >= 0 and < 1.')
if any(not np.isfinite(v) or abs(v)>1 for v in interventions.values()): raise ValueError('Inputs must be finite, between -1 and 1.')
trajectory = simulate(selected, interventions, steps, pulse, gain)
baseline = simulate(selected, {}, steps, pulse, gain)
fig = px.line(trajectory, title='Perturbation and recovery · arbitrary units', labels={'value':'Model response','step':'Simulation step'})
fig.add_vline(x=pulse, line_dash='dash', annotation_text='Last driven state; next update has no input')
fig.show()
display(trajectory.tail())
print('Baseline response:', float(np.abs(baseline.values).max()))
print('Model: x[t+1] = tanh(gain × normalized_W × x[t] + input[t]). Input stops after pulse steps.')

## Layer 09 · Uncertainty Resolver
Quantifies directional disagreement, without resolving which observation is correct. Disagreement is not a calibrated uncertainty interval. Zero-confidence records carry no directional information.

In [ ]:
banner(9, 'Uncertainty Resolver')
uncertainty = edges.sort_values('disagreement', ascending=False).copy()
display(uncertainty)
px.scatter(uncertainty, x='confidence', y='disagreement', size='evidence_count', color='disagreement',
           hover_data=['source','target'], color_continuous_scale='Plasma', title='Evidence confidence and directional conflict').show()
print('Disagreement = 1 - abs(sum(effect × confidence)) / sum(confidence); zero when total confidence is zero.')

## Layer 10 · Next-Best-Evidence Selector
Ranks existing links using an explicit heuristic. It does not optimize experiment cost, feasibility or expected information gain, and does not discover unmeasured links.

In [ ]:
banner(10, 'Next-Best-Evidence Selector')
priorities = edges.copy()
priorities['priority'] = (1-priorities.confidence+priorities.disagreement)/(1+priorities.evidence_count)
priorities = priorities.sort_values('priority',ascending=False)
display(priorities)
px.bar(priorities.head(15).assign(link=lambda d:d.source+' → '+d.target),x='priority',y='link',orientation='h',
       color='priority',color_continuous_scale='Viridis',title='Heuristic evidence priorities').show()
print('Priority = (1 - mean confidence + disagreement) / (1 + record count).')

## Layer 11 · Cross-Context Transfer Test
This prototype provides descriptive edge overlap and directional comparison, not predictive transfer validation. Choose a different observed joint context.

In [ ]:
banner(11, 'Cross-Context Transfer Test')
available_contexts = evidence[CONTEXT].drop_duplicates().reset_index(drop=True)
display(available_contexts)
alternatives = [r for r in available_contexts.to_dict('records') if r != context]
comparison_context = alternatives[0] if alternatives else context.copy()
# Override comparison_context with another tissue/species/disease combination if desired.
other = aggregate(select(evidence, comparison_context, threshold))
comparison = edges.merge(other, on=['source','target'], how='outer', suffixes=('_current','_comparison'), indicator=True)
shared = comparison['_merge'].eq('both')
comparison['opposite_nonzero_signs'] = shared & (comparison.weight_current * comparison.weight_comparison < 0)
jaccard = float(shared.sum()/len(comparison)) if len(comparison) else None
display(comparison)
print('Comparison context:', comparison_context)
print('Directed edge Jaccard overlap:', jaccard)
if comparison_context == context: print('Only a same-context comparison: choose another context when available.')
if other.empty: print('Comparison has no evidence; overlap cannot establish biological differences.')

## Layer 12 · Reproducible Evidence Package
Collects results from all preceding layers, source code, parameters, evidence and software environment into one ZIP. The notebook can display a download link without starting Streamlit.

In [ ]:
banner(12, 'Reproducible Evidence Package')
import subprocess
from IPython.display import FileLink
params = {'interventions':interventions,'steps':steps,'pulse':pulse,'gain':gain,'threshold':threshold,
          'study_metadata':study_metadata,'context_fingerprint':context_fingerprint,
          'path_source':source,'path_target':target,'max_hops':max_hops,'comparison_context':comparison_context}
buffer = io.BytesIO(package(selected, context, params, trajectory))
with zipfile.ZipFile(buffer, 'a', zipfile.ZIP_DEFLATED) as archive:
    tables = {'all_evidence':evidence,'entities':entity_table,'validation':validation_summary,'provenance':provenance,
              'bridges':bridges,'asymmetry':asymmetry,'mechanisms':mechanisms,'uncertainty':uncertainty,
              'next_evidence':priorities,'context_comparison':comparison,'baseline':baseline}
    for name, table in tables.items(): archive.writestr(name+'.csv',table.to_csv(index=name=='baseline'))
    for name in ['core.py','app.py','requirements.txt','README.md','.streamlit/config.toml']:
        archive.writestr('source/'+name,Path(name).read_text())
    archive.writestr('environment.txt',subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
output = Path('EENP_All_12_Layers_Results.zip')
output.write_bytes(buffer.getvalue())
print('Saved:',output)
display(FileLink(str(output)))
# In Colab you can also download this ZIP from the Files panel on the left.

## Optional · Launch Streamlit
All 12 Colab layers above work without this step. Set LAUNCH_STREAMLIT=True only if you want the separate app. Add NGROK_AUTHTOKEN to Colab Secrets first. The URL is temporary and public; use non-sensitive data. Default False keeps Runtime → Run all usable without a token.

In [ ]:
LAUNCH_STREAMLIT = False
if LAUNCH_STREAMLIT:
    from google.colab import userdata
    from pyngrok import ngrok
    import subprocess, time, urllib.request
    token = userdata.get("NGROK_AUTHTOKEN")
    if not token: raise ValueError("Add NGROK_AUTHTOKEN to Colab Secrets")
    ngrok.set_auth_token(token)
    del token
    if "server" in globals() and server.poll() is None:
        server.terminate()
        server.wait(timeout=15)
    ngrok.kill()
    log = open("streamlit.log", "w")
    server = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.headless=true"], stdout=log, stderr=subprocess.STDOUT)
    ready = False
    for _ in range(40):
        if server.poll() is not None: break
        try:
            with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=1) as response:
                ready = response.status == 200
            if ready: break
        except Exception: time.sleep(0.5)
    if not ready:
        raise RuntimeError(Path("streamlit.log").read_text()[-4000:])
    tunnel = ngrok.connect(8501, "http")
    print("Open your temporary Streamlit app:", tunnel.public_url)
else:
    print("Skipped optional Streamlit launch. All 12 layers are available above.")

## Optional · Stop Streamlit
Change STOP_STREAMLIT to True and run when finished using the app.

In [ ]:
STOP_STREAMLIT = False
if STOP_STREAMLIT:
    from pyngrok import ngrok
    ngrok.kill()
    if "server" in globals() and server.poll() is None:
        server.terminate()
        server.wait(timeout=15)
    if "log" in globals(): log.close()
    print("App and tunnel stopped.")